# EdgeVision — Colab Training Pipeline

Trains **MobileNetV2**, **ShuffleNetV2**, and **ResNet18** on CIFAR-10,
then saves models + benchmarks inside the Colab environment.

**After training**, download the `models/` folder and copy it into your
local `EdgeVision/EdgeVision/models/` directory, then commit to GitHub.

---
## Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells (takes ~15-20 min)
3. Download `models/` folder and place in your `EdgeVision/EdgeVision/` repo

---
## 1. Setup

In [ ]:
# @title Install dependencies
import torch
has_gpu = torch.cuda.is_available()
if not has_gpu:
    !pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy pandas scikit-learn matplotlib pillow onnx onnxruntime
import torch; print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

In [ ]:
# @title Clone EdgeVision repo
import os
if not os.path.exists("/content/EdgeVision/app.py"):
    !git clone https://github.com/bravo2024/EdgeVision.git /content/EdgeVision
else:
    print("Already cloned")
%cd /content/EdgeVision
!echo "Working in: $(pwd)"

---
## 2. Data

In [ ]:
# @title Load CIFAR-10 + verify GPU
import torch, sys; sys.path.insert(0, "/content/EdgeVision")
from src.data import load_cifar10, make_torch_loaders

data = load_cifar10()
print(f"Train: {data['n_train']} images")
print(f"Test:  {data['n_test']} images")
print(f"Classes: {', '.join(data['class_names'])}")
print(f"Device: {'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | PyTorch {torch.__version__}")

---
## 3. Train Models

Each model trains for 20 epochs on GPU (~3-5 min each).

In [ ]:
# @title Train MobileNetV2
import torch
from pathlib import Path
from src.model import build_mobilenet_v2, train_model, evaluate_model, benchmark_model

device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader, test_loader = make_torch_loaders(
    data["train_images"], data["train_labels"],
    data["test_images"], data["test_labels"],
    batch_size=128
)

model = build_mobilenet_v2(10)
result = train_model(model, train_loader, test_loader, num_epochs=20, lr=0.1, device=device)
eval_res = evaluate_model(result["model"], test_loader, data["class_names"], device=device)
bench = benchmark_model(result["model"], test_loader, device=device)

print(f"Accuracy: {eval_res['accuracy']:.4f}")
print(f"Size: {bench['model_size_mb']:.2f} MB | Latency: {bench['avg_inference_ms']:.1f} ms")

import pickle
Path("models").mkdir(exist_ok=True)
pickle.dump(result["model"], open("models/mobilenet_v2.pth", "wb"))
print("Saved: models/mobilenet_v2.pth")

In [ ]:
# @title Train ShuffleNetV2
from src.model import build_shufflenet_v2

model = build_shufflenet_v2(10)
result = train_model(model, train_loader, test_loader, num_epochs=20, lr=0.1, device=device)
eval_res = evaluate_model(result["model"], test_loader, data["class_names"], device=device)
bench = benchmark_model(result["model"], test_loader, device=device)

print(f"Accuracy: {eval_res['accuracy']:.4f}")
print(f"Size: {bench['model_size_mb']:.2f} MB | Latency: {bench['avg_inference_ms']:.1f} ms")

pickle.dump(result["model"], open("models/shufflenet_v2.pth", "wb"))
print("Saved: models/shufflenet_v2.pth")

In [ ]:
# @title Train ResNet18
from src.model import build_resnet18

model = build_resnet18(10, pretrained=False)
result = train_model(model, train_loader, test_loader, num_epochs=20, lr=0.1, device=device)
eval_res = evaluate_model(result["model"], test_loader, data["class_names"], device=device)
bench = benchmark_model(result["model"], test_loader, device=device)

print(f"Accuracy: {eval_res['accuracy']:.4f}")
print(f"Size: {bench['model_size_mb']:.2f} MB | Latency: {bench['avg_inference_ms']:.1f} ms")

pickle.dump(result["model"], open("models/resnet18.pth", "wb"))
print("Saved: models/resnet18.pth")

---
## 4. Export & Quantize

In [ ]:
# @title ONNX Export
from src.model import export_onnx

for name in ["mobilenet_v2", "shufflenet_v2", "resnet18"]:
    model = pickle.load(open(f"models/{name}.pth", "rb"))
    path = f"models/{name}.onnx"
    export_onnx(model.cpu(), path)
    print(f"Exported: {path}")

In [ ]:
# @title INT8 Quantization
from src.model import quantize_model

for name in ["mobilenet_v2", "shufflenet_v2", "resnet18"]:
    model = pickle.load(open(f"models/{name}.pth", "rb"))
    quantized = quantize_model(model.cpu())
    pickle.dump(quantized, open(f"models/{name}_quantized.pth", "wb"))
    orig_size = sum(p.numel() for p in model.parameters()) * 4 / (1024 * 1024)
    q_size = sum(p.numel() for p in quantized.parameters()) * 1 / (1024 * 1024)
    print(f"{name}: {orig_size:.2f} MB -> {q_size:.2f} MB")

In [ ]:
# @title Build benchmark cache (for Streamlit app)
import json

benchmarks = {}
val_accs = {}
display_names = {
    "mobilenet_v2": "MobileNetV2",
    "shufflenet_v2": "ShuffleNetV2",
    "resnet18": "ResNet18",
}

for name in ["mobilenet_v2", "shufflenet_v2", "resnet18"]:
    model = pickle.load(open(f"models/{name}.pth", "rb"))
    bench = benchmark_model(model, test_loader, device=device)
    eval_res = evaluate_model(model, test_loader, data["class_names"], device=device)
    display = display_names[name]
    benchmarks[display] = bench
    val_accs[display] = eval_res["accuracy"]

cache = {
    "benchmarks": {k: {sk: sv for sk, sv in v.items() if sk != "predictions"} for k, v in benchmarks.items()},
    "val_accs": val_accs,
}

with open("models/benchmark_cache.json", "w") as f:
    json.dump(cache, f, indent=2)

print("Saved: models/benchmark_cache.json")
for k, v in val_accs.items():
    print(f"  {k}: {v:.4f}")

---
## 5. Download Models

All files are in `/content/EdgeVision/models/`.

In [ ]:
# @title List all generated files
from pathlib import Path
print(f"{'File':45s} {'Size':>12s}")
print("-" * 58)
for f in sorted(Path("models").iterdir()):
    sz = f.stat().st_size
    if sz > 1024 * 1024:
        print(f"{f.name:45s} {sz / 1024 / 1024:>8.2f} MB")
    elif sz > 1024:
        print(f"{f.name:45s} {sz / 1024:>8.1f} KB")
    else:
        print(f"{f.name:45s} {sz:>8d} B")

In [ ]:
# @title Zip models/ for easy download
!zip -r /content/models.zip models/
!ls -lh /content/models.zip
print("\nDownload the zip or individual files from Colab's file browser.")

---
## How to use these models

1. **Download** `models.zip` from Colab (or grab individual files)
2. **Extract** into your local `EdgeVision/EdgeVision/models/` directory
3. **Commit** to GitHub:
   ```
   git add models/ && git commit -m "trained models from Colab" && git push
   ```
4. **Streamlit Cloud** auto-redeploys and loads cached benchmarks immediately